In [ ]:
"""
MIT 8334 Capstone — Group 3: Historical Weather and Climate Variability in Kenya
ONE-FILE PIPELINE: pulls real data from Open-Meteo, runs the Data Quality
Assessment, cleans it, and writes the final CSV — all in a single run.

HOW TO RUN (pick one):

Option A — Google Colab (easiest, no install, works from a browser):
  1. Go to https://colab.research.google.com/ and create a new notebook.
  2. Paste this entire file into one cell.
  3. Run the cell. It installs nothing extra (pandas/numpy/requests are
     already in Colab). Files appear in the Colab file browser on the left
     under /content/data/ — download them from there.

Option B — Your own machine:
  1. pip install pandas numpy requests
  2. python3 group3_pipeline.py
  3. Look in ./data/raw/ and ./data/cleaned/ and ./DATA_QUALITY_ASSESSMENT.md

Nothing else needs to be run before or after this. This one file replaces
data_acquisition.py + data_quality_cleaning.py from the earlier version.
"""

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# ===========================================================================
# CONFIG
# ===========================================================================
LOCATIONS = {
    "Nairobi": (-1.2921, 36.8219),
    "Kisumu": (-0.0917, 34.7680),
    "Mombasa": (-4.0435, 39.6682),
    "Eldoret": (0.5143, 35.2698),
    "Nakuru": (-0.3031, 36.0800),
    "Garissa": (-0.4536, 39.6401),
}

START_DATE = "2015-01-01"
END_DATE = "2024-12-31"

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

DAILY_VARIABLES = [
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "precipitation_sum",
    "rain_sum",
    "windspeed_10m_max",
]

PLAUSIBLE_RANGES = {
    "temperature_2m_max": (5, 45),
    "temperature_2m_min": (-5, 35),
    "temperature_2m_mean": (0, 40),
    "precipitation_sum": (0, 300),
    "rain_sum": (0, 300),
    "windspeed_10m_max": (0, 120),
}

RAW_DIR = Path("data/raw")
CLEAN_DIR = Path("data/cleaned")
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

report_lines = []


def log(line=""):
    print(line)
    report_lines.append(line)


# ===========================================================================
# STEP 1 — ACQUISITION
# ===========================================================================
def fetch_location(lat, lon, max_retries=6):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ",".join(DAILY_VARIABLES),
        "timezone": "Africa/Nairobi",
    }
    for attempt in range(max_retries):
        response = requests.get(BASE_URL, params=params, timeout=30)
        if response.status_code == 429:
            wait = 15 * (attempt + 1)  # 15s, 30s, 45s, ... backoff
            print(f"  Rate limited (429). Waiting {wait}s before retry "
                  f"({attempt + 1}/{max_retries})...")
            time.sleep(wait)
            continue
        response.raise_for_status()
        return response.json()
    raise RuntimeError("Gave up after repeated 429 rate-limit responses. "
                        "Wait a few minutes and rerun.")


def acquire_all():
    combined = {}
    for name, (lat, lon) in LOCATIONS.items():
        out_path = RAW_DIR / f"{name.lower()}.json"
        if out_path.exists():
            # Already fetched in a previous run (e.g. before a rate-limit
            # error) — skip it so a rerun resumes instead of starting over.
            print(f"Skipping {name}, already saved at {out_path}.")
            with open(out_path) as f:
                combined[name] = json.load(f)
            continue
        print(f"Fetching {name} ({lat}, {lon})...")
        data = fetch_location(lat, lon)
        with open(out_path, "w") as f:
            json.dump(data, f, indent=2)
        combined[name] = data
        time.sleep(5)
    with open(RAW_DIR / "all_locations_combined.json", "w") as f:
        json.dump(combined, f, indent=2)
    print("Acquisition complete.\n")


# ===========================================================================
# STEP 2 — LOAD INTO ONE DATAFRAME
# ===========================================================================
def load_all_locations():
    frames = []
    for loc in LOCATIONS:
        with open(RAW_DIR / f"{loc.lower()}.json") as f:
            data = json.load(f)
        daily = data["daily"]
        df = pd.DataFrame({"date": daily["time"]})
        for var in DAILY_VARIABLES:
            df[var] = daily.get(var)
        df["location"] = loc
        frames.append(df)
    combined = pd.concat(frames, ignore_index=True)
    combined["date"] = pd.to_datetime(combined["date"])
    return combined


# ===========================================================================
# STEP 3 — DATA QUALITY ASSESSMENT
# ===========================================================================
def assess_quality(df, expected_start, expected_end):
    log("# Data Quality Assessment\n")
    log(f"Raw rows loaded: {len(df)}")
    log(f"Locations: {df['location'].nunique()} ({', '.join(sorted(df['location'].unique()))})\n")

    log("## Missing values (NaNs in existing rows)")
    for var, count in df[DAILY_VARIABLES].isnull().sum().items():
        log(f"- {var}: {count}")
    log("")

    log("## Missing dates (gaps in the expected daily sequence)")
    expected_dates = pd.date_range(expected_start, expected_end, freq="D")
    for loc in LOCATIONS:
        loc_dates = set(df.loc[df["location"] == loc, "date"])
        missing = sorted(set(expected_dates) - loc_dates)
        pct = 100 * len(missing) / len(expected_dates)
        log(f"- {loc}: {len(missing)} missing day(s) ({pct:.2f}% of expected range)")
    log("")

    log("## Duplicate records (same location + date)")
    dup_mask = df.duplicated(subset=["location", "date"], keep=False)
    log(f"- {dup_mask.sum()} duplicate row(s) found across all locations")
    if dup_mask.sum():
        for loc, count in df[dup_mask].groupby("location").size().items():
            log(f"  - {loc}: {count} duplicate row(s)")
    log("")

    log("## Outliers (values outside physically plausible ranges for Kenya)")
    outlier_flags = pd.DataFrame(index=df.index)
    for var, (low, high) in PLAUSIBLE_RANGES.items():
        outlier_flags[var] = ~df[var].between(low, high) & df[var].notna()
    any_outlier = outlier_flags.any(axis=1)
    log(f"- {any_outlier.sum()} row(s) contain at least one implausible value")
    for var in PLAUSIBLE_RANGES:
        n = outlier_flags[var].sum()
        if n:
            log(f"  - {var}: {n} value(s) outside plausible range {PLAUSIBLE_RANGES[var]}")
    log("")

    log("## Data completeness (% of expected days actually present, per location)")
    for loc in LOCATIONS:
        n_present = (df["location"] == loc).sum()
        pct = 100 * n_present / len(expected_dates)
        log(f"- {loc}: {n_present}/{len(expected_dates)} days ({pct:.2f}%)")
    log("")

    log("## Consistency across locations")
    for var, dtype in df[DAILY_VARIABLES].dtypes.items():
        log(f"- {var}: {dtype}")
    date_span_ok = all(
        df.loc[df["location"] == loc, "date"].between(expected_start, expected_end).all()
        for loc in LOCATIONS
    )
    log(f"All dates fall within {expected_start.date()}–{expected_end.date()}: {date_span_ok}\n")

    return outlier_flags


# ===========================================================================
# STEP 4 — CLEANING
# ===========================================================================
def clean_data(df, expected_start, expected_end):
    log("## Cleaning decisions applied\n")
    df = df.copy()

    before = len(df)
    df = df.drop_duplicates(subset=["location", "date"], keep="first").reset_index(drop=True)
    log(f"- Dropped {before - len(df)} duplicate row(s), kept first occurrence. "
        f"Duplicates would double-count those days in aggregation.")

    outlier_flags = pd.DataFrame(index=df.index)
    for var, (low, high) in PLAUSIBLE_RANGES.items():
        outlier_flags[var] = ~df[var].between(low, high) & df[var].notna()
    n_outliers = 0
    for var in PLAUSIBLE_RANGES:
        n = outlier_flags[var].sum()
        n_outliers += n
        df.loc[outlier_flags[var], var] = np.nan
    log(f"- Converted {n_outliers} physically implausible value(s) to missing rather than "
        f"dropping the whole row, since the other variables that day are still usable.")

    expected_dates = pd.date_range(expected_start, expected_end, freq="D")
    reindexed = []
    for loc in LOCATIONS:
        loc_df = df[df["location"] == loc].set_index("date").reindex(expected_dates)
        loc_df["location"] = loc
        loc_df.index.name = "date"
        reindexed.append(loc_df)
    df = pd.concat(reindexed).reset_index().rename(columns={"index": "date"})
    log(f"- Reindexed every location to the full daily calendar so missing days become "
        f"explicit rows rather than silently absent.")

    max_gap_days = 3
    for loc in LOCATIONS:
        mask = df["location"] == loc
        for var in DAILY_VARIABLES:
            df.loc[mask, var] = (
                df.loc[mask, var].interpolate(method="linear", limit=max_gap_days, limit_direction="both")
            )
    remaining_na = df[DAILY_VARIABLES].isnull().sum().sum()
    log(f"- Linearly interpolated gaps of {max_gap_days} day(s) or fewer per location per "
        f"variable. Longer gaps are left as NaN rather than guessed. "
        f"{remaining_na} value(s) remain missing.")

    log("- Schema and units were already consistent across locations (°C, mm, km/h); "
        "only column order/naming was standardized.\n")

    return df[["date", "location"] + DAILY_VARIABLES]


# ===========================================================================
# MAIN
# ===========================================================================
def main():
    expected_start = pd.Timestamp(START_DATE)
    expected_end = pd.Timestamp(END_DATE)

    acquire_all()

    df = load_all_locations()
    assess_quality(df, expected_start, expected_end)
    cleaned = clean_data(df, expected_start, expected_end)

    out_csv = CLEAN_DIR / "kenya_weather_cleaned.csv"
    cleaned.to_csv(out_csv, index=False)
    log(f"Cleaned dataset exported to {out_csv} ({len(cleaned)} rows, "
        f"{cleaned['location'].nunique()} locations).")

    with open("DATA_QUALITY_ASSESSMENT.md", "w") as f:
        f.write("\n".join(report_lines))
    print("\nDone. Check ./data/raw/, ./data/cleaned/, and ./DATA_QUALITY_ASSESSMENT.md")


if __name__ == "__main__":
    main()

Skipping Nairobi, already saved at data/raw/nairobi.json.
Skipping Kisumu, already saved at data/raw/kisumu.json.
Skipping Mombasa, already saved at data/raw/mombasa.json.
Skipping Eldoret, already saved at data/raw/eldoret.json.
Fetching Nakuru (-0.3031, 36.08)...
Fetching Garissa (-0.4536, 39.6401)...
Acquisition complete.

# Data Quality Assessment

Raw rows loaded: 21918
Locations: 6 (Eldoret, Garissa, Kisumu, Mombasa, Nairobi, Nakuru)

## Missing values (NaNs in existing rows)
- temperature_2m_max: 0
- temperature_2m_min: 0
- temperature_2m_mean: 0
- precipitation_sum: 0
- rain_sum: 0
- windspeed_10m_max: 0

## Missing dates (gaps in the expected daily sequence)
- Nairobi: 0 missing day(s) (0.00% of expected range)
- Kisumu: 0 missing day(s) (0.00% of expected range)
- Mombasa: 0 missing day(s) (0.00% of expected range)
- Eldoret: 0 missing day(s) (0.00% of expected range)
- Nakuru: 0 missing day(s) (0.00% of expected range)
- Garissa: 0 missing day(s) (0.00% of expected range)

#

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os

os.makedirs('group3_outputs', exist_ok=True)
shutil.copytree('data', 'group3_outputs/data')
shutil.copy('DATA_QUALITY_ASSESSMENT.md', 'group3_outputs/')

shutil.make_archive('group3_outputs', 'zip', 'group3_outputs')

'/content/group3_outputs.zip'